# RAG Tuning Guide: Variables You Can Control

As a RAG application developer, you don't train a model from scratch.
But you control **many variables** that determine how well your system performs.

This guide covers:
1. What variables exist
2. What each one does
3. How to set them for different use cases (coding agent, medical, customer service)

## 1. The Full List of Tunable Variables

```
┌─────────────────────────────────────────────────────────────────┐
│                    RAG PIPELINE                                  │
│                                                                 │
│  [Documents] → [Chunking] → [Embedding] → [Vector DB]          │
│                                                                 │
│  [Query] → [Embedding] → [Search] → [Rerank] → [LLM Generate] │
│                                                                 │
│  Variables at each stage:                                       │
│                                                                 │
│  Chunking:     chunk_size, chunk_overlap, strategy              │
│  Embedding:    model, dimensions, batch_size                    │
│  Search:       top_k, similarity_threshold, search_type         │
│  Reranking:    rerank_model, rerank_top_n                       │
│  Generation:   temperature, max_tokens, system_prompt           │
│  Architecture: hybrid_weight, metadata_filters, decomposition   │
└─────────────────────────────────────────────────────────────────┘
```

## 2. Each Variable Explained

In [ ]:
# All tunable variables in a RAG system

variables = {
    "CHUNKING": [
        {
            "name": "chunk_size",
            "what": "How many tokens per chunk",
            "range": "100 - 2000 tokens",
            "tradeoff": "Small = precise retrieval, Large = more context per chunk",
        },
        {
            "name": "chunk_overlap",
            "what": "How many tokens overlap between adjacent chunks",
            "range": "0 - 50% of chunk_size",
            "tradeoff": "More overlap = no info lost at boundaries, but more storage",
        },
        {
            "name": "chunking_strategy",
            "what": "How to split: fixed size, by sentence, by paragraph, by heading",
            "range": "fixed | sentence | paragraph | semantic | recursive",
            "tradeoff": "Semantic = best quality, Fixed = simplest and cheapest",
        },
    ],
    "EMBEDDING": [
        {
            "name": "embedding_model",
            "what": "Which model converts text to vectors",
            "range": "text-embedding-3-small/large, BGE, E5, Cohere",
            "tradeoff": "Larger model = better quality, slower, more expensive",
        },
        {
            "name": "embedding_dimensions",
            "what": "Size of each vector (if model supports dimension reduction)",
            "range": "256 - 3072",
            "tradeoff": "Higher = captures more nuance, Lower = faster search, less storage",
        },
    ],
    "SEARCH": [
        {
            "name": "top_k",
            "what": "How many chunks to retrieve",
            "range": "1 - 20",
            "tradeoff": "More = comprehensive but noisy, Fewer = focused but might miss",
        },
        {
            "name": "similarity_threshold",
            "what": "Minimum cosine similarity to include a result",
            "range": "0.0 - 1.0",
            "tradeoff": "High = only very relevant, Low = cast wider net",
        },
        {
            "name": "search_type",
            "what": "Vector only, keyword only, or hybrid",
            "range": "vector | keyword (BM25) | hybrid",
            "tradeoff": "Hybrid = best quality, Vector = best for semantic, Keyword = exact match",
        },
        {
            "name": "hybrid_weight",
            "what": "Balance between vector and keyword (in hybrid mode)",
            "range": "0.0 (all keyword) - 1.0 (all vector)",
            "tradeoff": "More vector = semantic meaning, More keyword = exact terms",
        },
    ],
    "GENERATION": [
        {
            "name": "temperature",
            "what": "How creative/random the LLM response is",
            "range": "0.0 - 2.0",
            "tradeoff": "Low = deterministic/factual, High = creative/varied",
        },
        {
            "name": "top_k (generation)",
            "what": "How many token candidates the LLM considers at each step",
            "range": "1 - 100",
            "tradeoff": "Low = safe/predictable, High = diverse/surprising",
        },
        {
            "name": "max_tokens",
            "what": "Maximum length of generated answer",
            "range": "50 - 4000+",
            "tradeoff": "Longer = more thorough, Shorter = concise and cheaper",
        },
    ],
}

print("ALL TUNABLE VARIABLES IN A RAG SYSTEM")
print("=" * 75)

for category, vars_list in variables.items():
    print(f"\n  [{category}]")
    print(f"  {'-' * 70}")
    for v in vars_list:
        print(f"  {v['name']:<25} {v['what']}")
        print(f"  {'':25} Range: {v['range']}")
        print(f"  {'':25} Tradeoff: {v['tradeoff']}")
        print()

## 3. Use Case: Coding Agent

**Goal:** Find relevant code snippets, API docs, error solutions.  
**Priority:** Precision — find the RIGHT function/class, ignore similar-looking but wrong matches.  
**Challenge:** Code has strict syntax — `setTimeout` is NOT `setInterval`, even though they're semantically close.

In [ ]:
coding_agent_config = {
    # CHUNKING
    "chunk_size": 500,           # Smaller: one function per chunk
    "chunk_overlap": 50,         # Small overlap: function boundaries are clear
    "chunking_strategy": "semantic (split by function/class definition)",
    
    # EMBEDDING
    "embedding_model": "text-embedding-3-large or code-specific (CodeBERT, StarCoder-embed)",
    "embedding_dimensions": 1536,  # High: code has many subtle distinctions
    
    # SEARCH
    "top_k": 5,                  # Moderate: show a few options
    "similarity_threshold": 0.75, # HIGH: only very relevant code
    "search_type": "hybrid",
    "hybrid_weight": 0.4,        # More keyword (0.6): exact function names matter!
    
    # GENERATION
    "temperature": 0.0,          # ZERO: code must be exact, no creativity
    "max_tokens": 2000,          # Long: full code examples
}

print("CODING AGENT — Configuration")
print("=" * 65)
print(f"\n  Priority: PRECISION (right function > vaguely related code)")
print(f"  Temperature: 0.0 (code must be syntactically correct)")
print(f"  Keyword weight: HIGH (exact names like 'useState' matter)")
print()

for key, val in coding_agent_config.items():
    print(f"  {key:<25} = {val}")

print(f"\n  WHY these values:")
print(f"  ─────────────────")
print(f"  • chunk_size=500: one function fits in one chunk.")
print(f"    A 2000-token chunk mixing 4 functions dilutes relevance.")
print(f"")
print(f"  • hybrid_weight=0.4 (60% keyword): 'React.useState' must match")
print(f"    EXACTLY, not just semantically similar hooks.")
print(f"    Vector search alone might return useReducer or useRef.")
print(f"")
print(f"  • similarity_threshold=0.75: if nothing scores above 0.75,")
print(f"    say 'not found' rather than showing wrong code.")
print(f"    Wrong code is worse than no code.")
print(f"")
print(f"  • temperature=0.0: generated code must compile.")
print(f"    Creativity = bugs in this context.")

In [ ]:
# Example: coding agent search

print("CODING AGENT — Example Query")
print("=" * 65)
print()
print('  Query: "How to debounce input in React?"')
print()
print("  Step 1 — Hybrid search (keyword=0.6, vector=0.4):")
print("    Keywords extracted: 'debounce', 'input', 'React'")
print()
print("    Results:")
print("    ┌───────────────────────────────────────────────────────────┐")
print("    │ #1 [0.92] hooks/useDebounce.ts                           │")
print("    │    function useDebounce(value, delay) { ... }            │")
print("    │    → EXACT match: 'debounce' keyword + semantic match    │")
print("    │                                                           │")
print("    │ #2 [0.81] utils/debounce.ts                              │")
print("    │    export function debounce(fn, ms) { ... }              │")
print("    │    → Has 'debounce' but generic (not React-specific)     │")
print("    │                                                           │")
print("    │ #3 [0.68] hooks/useThrottle.ts         ← EXCLUDED        │")
print("    │    Below threshold 0.75 — similar concept, wrong function│")
print("    └───────────────────────────────────────────────────────────┘")
print()
print("  Without high keyword weight, #3 (useThrottle) might rank higher")
print("  because semantically 'throttle' is close to 'debounce'.")
print("  But a developer asking for debounce does NOT want throttle.")

## 4. Use Case: Medical Information

**Goal:** Retrieve exact dosages, contraindications, drug interactions.  
**Priority:** SAFETY — wrong answer can harm patients. Better to say "I don't know" than guess.  
**Challenge:** "Metformin 500mg" and "Metformin 2550mg" are different prescriptions, not paraphrases.

In [ ]:
medical_config = {
    # CHUNKING
    "chunk_size": 300,           # SMALL: one guideline point per chunk
    "chunk_overlap": 100,        # HIGH overlap: don't lose context at boundaries
    "chunking_strategy": "paragraph (each guideline bullet = one chunk)",
    
    # EMBEDDING
    "embedding_model": "domain-specific (PubMedBERT, BioLORD) or fine-tuned general",
    "embedding_dimensions": 1024,  # High: medical terms have precise meanings
    
    # SEARCH
    "top_k": 10,                 # HIGH: retrieve many, then filter strictly
    "similarity_threshold": 0.6,  # MODERATE: cast wide net, rely on reranking
    "search_type": "hybrid",
    "hybrid_weight": 0.3,        # VERY high keyword (0.7): exact drug names, dosages
    "reranker": "cross-encoder (BGE-reranker-large or Cohere rerank)",
    "rerank_top_n": 3,           # After reranking, use only top 3
    
    # GENERATION
    "temperature": 0.0,          # ZERO: never be creative with medical info
    "max_tokens": 500,           # Short: concise factual answers
    
    # SAFETY
    "confidence_threshold": 0.8,  # HIGH: refuse if unsure
    "require_citation": True,     # Every claim must cite source document
    "freshness_boost": True,      # Prefer recent guidelines
}

print("MEDICAL INFORMATION — Configuration")
print("=" * 65)
print(f"\n  Priority: SAFETY (refuse > guess, exact > approximate)")
print(f"  Strategy: Wide retrieval → strict reranking → forced citation")
print()

for key, val in medical_config.items():
    print(f"  {key:<25} = {val}")

print(f"\n  WHY these values:")
print(f"  ─────────────────")
print(f"  • chunk_size=300: one dosing rule per chunk.")
print(f"    'eGFR 30-44: reduce to 1000mg' must be its own retrievable unit.")
print(f"    Mixed into a larger chunk, it could be drowned out.")
print(f"")
print(f"  • hybrid_weight=0.3 (70% keyword): exact terms are critical.")
print(f"    'eGFR 30' and 'eGFR 45' are DIFFERENT thresholds.")
print(f"    Vector search treats them as similar (both about kidney function).")
print(f"    Keyword search distinguishes the exact numbers.")
print(f"")
print(f"  • top_k=10 + rerank_top_n=3: retrieve broadly, then filter strictly.")
print(f"    The cross-encoder reranker compares query+doc word-by-word,")
print(f"    catching exact matches that embedding search might miss.")
print(f"")
print(f"  • confidence_threshold=0.8: if the best result scores below 0.8,")
print(f"    the system REFUSES to answer. 'I don't know' is safer than")
print(f"    a confident wrong dosage.")
print(f"")
print(f"  • temperature=0.0: 'Metformin 1000mg' is not the same as")
print(f"    'Metformin 1050mg'. Zero creativity. Zero variation.")

In [ ]:
# Example: medical search

print("MEDICAL — Example Query")
print("=" * 65)
print()
print('  Query: "Maximum metformin dose for patient with eGFR 35?"')
print()
print("  Step 1 — Extract entities: drug=metformin, lab=eGFR, value=35")
print("  Step 2 — Metadata filter: drug='metformin'")
print("  Step 3 — Hybrid search (keyword=0.7):")
print("    Keywords: 'metformin', 'maximum', 'dose', 'eGFR', '35'")
print()
print("    Initial retrieval (top_k=10):")
print("    ┌────────────────────────────────────────────────────────┐")
print("    │ #1 [0.91] Metformin Use in Renal Impairment            │")
print("    │    'eGFR 30-44: reduce maximum dose to 1000mg/day'     │")
print("    │                                                        │")
print("    │ #2 [0.84] Metformin Prescribing Guidelines             │")
print("    │    'Maximum dose: 2550mg/day in divided doses'          │")
print("    │    ⚠ DANGER: this is the GENERAL max, not for eGFR 35!│")
print("    │                                                        │")
print("    │ #3 [0.72] Metformin Drug Interactions                  │")
print("    │    (less relevant to dosing question)                   │")
print("    └────────────────────────────────────────────────────────┘")
print()
print("    After reranking (cross-encoder):")
print("    #1 stays #1 — it contains 'eGFR 30-44' which includes 35")
print()
print("  Step 4 — Generate with forced citation:")
print("    'For eGFR 35 (range 30-44), maximum metformin dose is")
print("     1000mg/day. Monitor renal function every 3 months. [D006]'")
print()
print("  WITHOUT keyword-heavy search:")
print("    Vector-only might rank #2 (general guidelines) equally with #1")
print("    because both are 'about metformin dosing'.")
print("    If the LLM picks 2550mg from #2 → WRONG and DANGEROUS.")

## 5. Use Case: Customer Service

**Goal:** Find relevant FAQ articles, product info, troubleshooting guides.  
**Priority:** RECALL — find anything that MIGHT help. Better to show extra info than miss the answer.  
**Challenge:** Customers describe problems in vague, non-technical language. "My thing isn't working" could match many docs.

In [ ]:
customer_service_config = {
    # CHUNKING
    "chunk_size": 1000,          # LARGE: full FAQ answer in one chunk
    "chunk_overlap": 200,        # Good overlap: customer issues span topics
    "chunking_strategy": "by FAQ entry or by section heading",
    
    # EMBEDDING
    "embedding_model": "text-embedding-3-small (good enough, cheaper)",
    "embedding_dimensions": 512,   # Lower OK: less nuance needed than medical
    
    # SEARCH
    "top_k": 8,                  # HIGH: show many possibly-related articles
    "similarity_threshold": 0.4,  # LOW: cast very wide net
    "search_type": "hybrid",
    "hybrid_weight": 0.7,        # More vector (semantic): customers use informal language
    
    # GENERATION
    "temperature": 0.3,          # SLIGHT creativity: friendly, natural responses
    "max_tokens": 1000,          # Long: step-by-step troubleshooting
    
    # UX
    "confidence_threshold": 0.3,  # LOW: try to help even with partial matches
    "show_related": True,        # Show 'you might also need...' suggestions
    "escalate_threshold": 0.2,   # Only escalate if NOTHING matches at all
}

print("CUSTOMER SERVICE — Configuration")
print("=" * 65)
print(f"\n  Priority: RECALL (find anything helpful, generous matching)")
print(f"  Strategy: Wide search, low threshold, semantic-heavy, friendly tone")
print()

for key, val in customer_service_config.items():
    print(f"  {key:<25} = {val}")

print(f"\n  WHY these values:")
print(f"  ─────────────────")
print(f"  • chunk_size=1000: a full FAQ answer is one chunk.")
print(f"    Customer wants the complete solution, not a fragment.")
print(f"")
print(f"  • hybrid_weight=0.7 (70% vector, 30% keyword):")
print(f"    Customer says: 'my thing stopped working after the update'")
print(f"    FAQ title says: 'Troubleshooting connectivity after v3.2 upgrade'")
print(f"    Keywords don't overlap much, but MEANING does!")
print(f"    Vector search bridges this vocabulary gap.")
print(f"")
print(f"  • similarity_threshold=0.4: include anything possibly relevant.")
print(f"    A slightly-related FAQ is better than 'I can't help you.'")
print(f"    Customer satisfaction > precision.")
print(f"")
print(f"  • temperature=0.3: slightly warm for natural, friendly responses.")
print(f"    'I'd be happy to help!' not 'The solution is as follows:.'")
print(f"    But not too high — don't make up steps that don't exist.")
print(f"")
print(f"  • embedding_dimensions=512: customers don't use precise jargon.")
print(f"    'Can't connect' and 'connection failed' are the same thing.")
print(f"    Lower dimensions still capture this. Save cost and speed.")

In [ ]:
# Example: customer service search

print("CUSTOMER SERVICE — Example Query")
print("=" * 65)
print()
print('  Query: "my app keeps crashing after I updated it yesterday"')
print()
print("  Step 1 — Semantic search (vector_weight=0.7):")
print("    The query has NO technical terms. Keyword search would struggle.")
print("    But vector search understands the MEANING.")
print()
print("    Results (threshold=0.4, so we include partial matches):")
print("    ┌────────────────────────────────────────────────────────────┐")
print("    │ #1 [0.78] App crashes after v4.2 update — known issue      │")
print("    │    'Clear cache and reinstall. Fix coming in v4.2.1'       │")
print("    │                                                            │")
print("    │ #2 [0.65] How to roll back to a previous app version       │")
print("    │    'Settings > App > Version History > Restore'            │")
print("    │                                                            │")
print("    │ #3 [0.52] General troubleshooting: force close and restart │")
print("    │    'Hold power button, force close, restart device'        │")
print("    │                                                            │")
print("    │ #4 [0.44] Reporting a bug to the development team          │")
print("    │    'Send logs via Settings > Help > Send Report'           │")
print("    └────────────────────────────────────────────────────────────┘")
print()
print("  ALL 4 are shown to the customer (threshold=0.4 includes #4)")
print()
print("  Generated response (temperature=0.3):")
print("  ┌─────────────────────────────────────────────────────────────┐")
print("  │ Sorry to hear about the crashes! This is a known issue with │")
print("  │ yesterday's update. Here's what should fix it:              │")
print("  │                                                             │")
print("  │ 1. Clear your app cache (Settings > Storage > Clear Cache)  │")
print("  │ 2. If that doesn't work, try reinstalling the app           │")
print("  │ 3. A permanent fix is coming in v4.2.1 (ETA: this week)      │")
print("  │                                                             │")
print("  │ You can also roll back to the previous version:             │")
print("  │   Settings > App > Version History > Restore                │")
print("  │                                                             │")
print("  │ Let me know if you need more help! 😊                       │")
print("  └─────────────────────────────────────────────────────────────┘")
print()
print("  Medical system would NEVER show #3 and #4 (below 0.8 threshold).")
print("  But for customer service, showing extra options is HELPFUL.")

## 6. Side-by-Side Comparison

In [ ]:
print("COMPARISON: Same Variables, Different Values")
print("=" * 75)
print()
print(f"{'Variable':<25} {'Coding Agent':>15} {'Medical':>15} {'Customer Svc':>15}")
print(f"{'-'*25} {'-'*15} {'-'*15} {'-'*15}")

comparisons = [
    ("chunk_size",            "500",     "300",      "1000"),
    ("chunk_overlap",         "50",      "100",      "200"),
    ("chunking_strategy",     "function", "paragraph", "FAQ entry"),
    ("embedding_model",       "code-spec", "medical",  "general"),
    ("embedding_dimensions",  "1536",    "1024",     "512"),
    ("top_k",                 "5",       "10",       "8"),
    ("similarity_threshold",  "0.75",    "0.60",     "0.40"),
    ("hybrid_weight (vector)", "0.4",    "0.3",      "0.7"),
    ("reranker",              "optional", "required", "optional"),
    ("temperature",           "0.0",     "0.0",      "0.3"),
    ("confidence_threshold",  "0.75",    "0.80",     "0.30"),
    ("max_tokens",            "2000",    "500",      "1000"),
]

for var, coding, medical, cs in comparisons:
    print(f"{var:<25} {coding:>15} {medical:>15} {cs:>15}")

print(f"\n{'Strategy':<25} {'Precision':>15} {'Safety':>15} {'Recall':>15}")
print(f"{'Motto':<25} {'Right > Many':>15} {'Refuse > Guess':>15} {'Helpful > Exact':>15}")

## 7. Decision Flowchart: How to Choose Values

In [ ]:
print("DECISION FLOWCHART")
print("=" * 65)
print()
print("  Q1: Is a wrong answer DANGEROUS?")
print("  │")
print("  ├── YES (medical, legal, financial)")
print("  │   → temperature = 0.0")
print("  │   → confidence_threshold = HIGH (0.8+)")
print("  │   → require citations")
print("  │   → add reranker")
print("  │")
print("  └── NO (customer service, general chat)")
print("      → temperature = 0.2-0.5")
print("      → confidence_threshold = LOW (0.3)")
print("      → show 'related articles'")
print()
print("  Q2: Do EXACT TERMS matter more than MEANING?")
print("  │")
print("  ├── YES (code, medical dosages, legal clauses)")
print("  │   → hybrid_weight toward keyword (0.3-0.4 vector)")
print("  │   → smaller chunks (preserve exact context)")
print("  │")
print("  └── NO (customer queries, general Q&A)")
print("      → hybrid_weight toward vector (0.6-0.8 vector)")
print("      → larger chunks (full answers)")
print()
print("  Q3: Does the user use PRECISE or VAGUE language?")
print("  │")
print("  ├── PRECISE (developers, doctors, lawyers)")
print("  │   → higher similarity_threshold (0.7+)")
print("  │   → fewer results (top_k=3-5)")
print("  │   → higher embedding dimensions")
print("  │")
print("  └── VAGUE (customers, general public)")
print("      → lower similarity_threshold (0.3-0.5)")
print("      → more results (top_k=5-10)")
print("      → lower embedding dimensions (still works)")
print()
print("  Q4: How LONG is the source content?")
print("  │")
print("  ├── SHORT (code functions, FAQ bullets, drug guidelines)")
print("  │   → chunk_size = 200-500")
print("  │   → one logical unit per chunk")
print("  │")
print("  └── LONG (articles, manuals, legal documents)")
print("      → chunk_size = 800-1500")
print("      → higher overlap (200+)")
print("      → consider hierarchical chunking (summary + detail)")

## 8. The Variables You CANNOT Tune (Without Retraining)

Everything above is what you control as an **application developer**.

But some things are baked into the model itself:

In [ ]:
print("WHAT YOU CAN vs CANNOT TUNE")
print("=" * 65)
print()
print("  ┌─ YOU CONTROL (no training needed) ─────────────────────────┐")
print("  │ • Chunk size, overlap, strategy                            │")
print("  │ • Which embedding model to use (swap, don't train)         │")
print("  │ • top_k, threshold, hybrid weight                          │")
print("  │ • Temperature, max_tokens, system prompt                   │")
print("  │ • Metadata filters, reranking, query decomposition         │")
print("  │ • Data curation (what goes INTO the vector DB)             │")
print("  └────────────────────────────────────────────────────────────┘")
print()
print("  ┌─ REQUIRES FINE-TUNING (training) ──────────────────────────┐")
print("  │ • The embedding model's understanding of your domain       │")
print("  │   → Fine-tune with your own similar/dissimilar pairs       │")
print("  │   → e.g. make it understand that 'k8s' = 'Kubernetes'     │")
print("  │                                                            │")
print("  │ • The LLM's behavior and response style                    │")
print("  │   → Fine-tune to follow your citation format               │")
print("  │   → Fine-tune to refuse when uncertain                     │")
print("  │                                                            │")
print("  │ • Domain-specific understanding                            │")
print("  │   → Medical: 'eGFR 35' means something specific            │")
print("  │   → Legal: 'force majeure' has precise meaning             │")
print("  │   → Code: 'useState' is React-specific                     │")
print("  └────────────────────────────────────────────────────────────┘")
print()
print("  THE 80/20 RULE:")
print("  ─────────────────")
print("  80% of RAG quality comes from TUNING (no training needed):")
print("    → Right chunk size")
print("    → Right search strategy")
print("    → Good system prompt")
print("    → Clean, well-structured source data")
print()
print("  20% comes from TRAINING (fine-tuning):")
print("    → Custom embedding model for your domain")
print("    → Fine-tuned LLM for your response format")
print()
print("  START with tuning. Only fine-tune when tuning plateaus.")

## Summary

| Priority | Strategy | Key Variables |
|---|---|---|
| **Precision** (coding) | Right answer only | High threshold, keyword-heavy, small chunks, temp=0 |
| **Safety** (medical) | Refuse if unsure | Reranker, citations, confidence check, temp=0 |
| **Recall** (customer svc) | Find anything helpful | Low threshold, vector-heavy, large chunks, temp=0.3 |

### The most impactful variables (in order)

1. **chunk_size** — wrong size = wrong retrieval, everything downstream fails
2. **hybrid_weight** — exact terms vs meaning, depends entirely on your users
3. **similarity_threshold** — controls precision vs recall tradeoff
4. **temperature** — factual (0) vs natural (0.3+)
5. **top_k + reranker** — broad search + precise filter = best of both

### How to iterate

```
1. Start with defaults (chunk=500, top_k=5, threshold=0.5, hybrid=0.5)
2. Test with 20-30 real queries from your users
3. Find failures: "why did it return the wrong doc?"
4. Adjust ONE variable at a time
5. Re-test. Repeat.
```